# Supernovae Lab 1

 ### Goals:
 
 In this first lab, we will explore how to:
 - Determine the population of binaries that reach the end of their lives,

 - Identify their collapse mechanisms (electron capture vs. iron core-collapse),

 - Link them to the observed supernova types they produce,

 - Calculate their ejecta masses

 - Examine their main formation channels using also their van den Heuvel diagrams.

Our analysis will be based on post-processing an already completed POSYDON population run.

## 1. Managing Large Binary Populations and Identifying Stellar End-of-Life Stages

In your own research with POSYDON you will likely work with much larger populations than the small examples we have been running here. This is crucial for obtaining realistic statistical results for the populations you want to study. In this session, we will practice handling such large populations. To begin, we will review the columns stored in the POSYDON population output (saved in .h5 files) and examine the evolution of a specific binary system.

### Making a Copy and Loading Your Population

In [ ]:
%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import os
import posydon
import pandas as pd

shared_file = (
    Path.home()
    / "data"
    / "Populations"
    / "5K_pops"
    / "1e+00_Zsun_population.h5"
)

local_file = Path.cwd() / shared_file.name

shutil.copy2(shared_file, local_file)

For more info: https://posydon.org/POSYDON/latest/api_reference/posydon.popsyn.html#module-posydon.popsyn.synthetic_population

Let's start by reading the output file from a POSYDON run and exploring the main datasets stored in the HDF file. We will examine three key datasets—`history`, `oneline`, and `formation_channels`—to understand the information each contains and the insights that can be extracted from them related to SNe. We will begin by exploring the history dataset.

In [ ]:
df=pd.read_hdf(local_file,key='history')

Print out the columns saved in the `history` data frame of the population:

In [ ]:
df.columns

Examine the evolution of a specific system from your population, focusing on the columns of interest.

In [ ]:
# Let's see the evolution of one system
col = ['step_names','time','state','event','S1_state','S2_state','S1_mass','S2_mass','orbital_period','separation','S1_surface_h1','S1_surface_n14','S2_surface_h1','S2_surface_n14']
df.loc[3][col]

Above, you can see the evolution of a specific binary system. In this lab, we will focus on the final stages of stellar evolution by identifying the rows and columns that provide information about a star’s end of life.

In POSYDON, the end stages are marked in the `event` column with either `CC1` or `CC2`, indicating whether the collapse corresponds to the primary or secondary star. Immediately after an `event = CC1`, you will notice that the `step_names` column transitions to `step_SN`. Our first task is to establish how to systematically identify all `CC1` and `CC2` events across the population.

POSYDON distinguishes between two types of core-collapse events: `CC1` for primaries (star_1, the initially more massive star) and `CC2` for secondaries (star_2). For single stars, only `CC1` events occur, since they are treated as primaries. With this in mind, let’s find all the `CC1` and `CC2` events that arise from the different evolutionary channels.

**Disclaimer**: For historical reasons, POSYDON uses the events `CC1` and `CC2` to mark the end of a star’s evolution, even when no core-collapse supernova occurs. All systems that complete their evolution pass through one of these events, regardless of whether the final outcome is a `NS`, `BH`, `WD`, or `massless_remnant` (in the case of a pair-instability supernova, PISN).

### Analyzing a Population of Supernovae Using Masking Techniques

Let's create a dataframe containing only the entries where the `step_names` column is equal to `step_SN`. By applying this mask, we can extract the properties of each supernova event, including the remnant mass, the compact object type (black hole, neutron star, white dwarf, or a massless remnant in the case of PISN), the post-core-collapse binary state, and the time at which the supernova occurred since the formation of the binary, assuming a starburst population.


In [ ]:
# Identify the post-supernova (step_SN) rows corresponding to the core collapse
# of the primary star (CC1) and the secondary star (CC2).
# The actual supernova properties are stored in the `step_SN` row, while the
# previous row (`event.shift(1)`) tells us which star underwent core collapse.
post_CC1 = ((df['step_names'] == "step_SN") & (df['event'].shift(1) == "CC1"))
post_CC2 = ((df['step_names'] == "step_SN") & (df['event'].shift(1) == "CC2"))

# -----------------------------------------------------------------------------
# Extract the post-supernova properties for systems where the primary star
# (star 1) collapsed.
# We keep:
#   - time: time since binary formation
#   - state: binary state immediately after the supernova
#   - S1_state: compact object type (WD, NS, BH, etc.)
#   - S1_mass: compact remnant mass
# -----------------------------------------------------------------------------
df1 = df[["time", "state", "S1_state", "S1_mass"]][post_CC1]

# Rename columns to make their meaning explicit.
df1.rename(columns={
    'state': 'binary_state_postCC',
    'S1_state': 'stellar_state_postCC',
    'S1_mass': 'compact_object_mass'
}, inplace=True)

# Record which star produced the compact remnant.
df1["progenitor_star"] = 1

# -----------------------------------------------------------------------------
# Repeat the same procedure for systems where the secondary star (star 2)
# experienced core collapse.
# -----------------------------------------------------------------------------
df2 = df[["time", "state", "S2_state", "S2_mass"]][post_CC2]

df2.rename(columns={
    'state': 'binary_state_postCC',
    'S2_state': 'stellar_state_postCC',
    'S2_mass': 'compact_object_mass'
}, inplace=True)

# Record that the compact remnant originated from the secondary star.
df2["progenitor_star"] = 2

# -----------------------------------------------------------------------------
# Combine the core-collapse events from both stars into a single dataframe.
# Each row now corresponds to one compact-object formation event in the
# synthetic population.
# -----------------------------------------------------------------------------
df_synthetic = pd.concat([df1, df2], axis=0)

### Printing out the outcome of the synthetic population we just created!

Let’s display the outcome of the synthetic population we just created! The table below summarizes the state of the binary after the collapse of the primary and secondary stars, along with the explosion times of both events. It also shows whether the system was disrupted or remained bound following each core-collapse, and whether the resulting remnant originated from a merger product or from a single star. In addition, it identifies the type of compact object formed and its mass (WD, NS, BH, or a massless remnant in the case of a PISN at low metallicity). In the progenitor column, a value of 1 refers to the primary star, while 2 refers to the secondary (if present).

In [ ]:
#col_CC = ["time", "binary_state_postCC", "stellar_state_postCC", "compact_object_mass", "progenitor_star"]
df_synthetic.head(10)

Identify all NS and BH remnants that formed from the secondary components of binary systems in the synthetic population:

In [ ]:
# filter only NS and BH from progenitor star 2
mask = (df_synthetic["stellar_state_postCC"].isin(["NS", "BH"])) & (df_synthetic["progenitor_star"] == 2)
ns_bh_from_star2 = df_synthetic[mask]

# count NS and BH separately
count_ns = (ns_bh_from_star2["stellar_state_postCC"] == "NS").sum()
count_bh = (ns_bh_from_star2["stellar_state_postCC"] == "BH").sum()

print("From progenitor star 2:")
print("NS:", count_ns)
print("BH:", count_bh)


<div class="alert alert-success">

## Exercise: Mass distribution of the Black Holes

1. Find all black holes from initially single stars. (Replace the **??** values on the code below)
2. Plot the BH mass distribution
   
</div>

<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Hint</summary></b>

Find all the lines from df_synthetic where the `binary_state_postCC` column is equal to `initially_single_star` \& `stellar_state_postCC` is `BH`

    
</details>

In [ ]:
#STEP 1

mask_singles_BH = (??) & (df_synthetic["stellar_state_postCC"].isin(["BH"]))

bh_from_singles = df_synthetic[mask_singles_BH]

#STEP 2

### plot ###

<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Solution </summary></b>
```python
#STEP 1

mask_singles_BH = (df_synthetic["binary_state_postCC"].isin(["initially_single_star"])) & (df_synthetic["stellar_state_postCC"].isin(["BH"]))

bh_from_singles = df_synthetic[mask_singles_BH]

#STEP 2

bh_masses = bh_from_singles["compact_object_mass"]
plt.figure()
plt.hist(bh_masses, bins=10)
plt.xlabel(r"BH mass [$M_\odot$]")
plt.ylabel("Count")
plt.title("BH mass distribution (initially single stars)")
plt.show()
```
    
</details>

## 2. Exploring the `oneline` DataFrame

Next, we explore the `key="oneline"` DataFrame of the synthetic population, which stores quantities such as `SN_type`, `h1_mass_ej`, and `he4_mass_ej`. Here, `h1_mass_ej` and `he4_mass_ej` represent the total masses of hydrogen and helium in the supernova ejecta, respectively.

Printing out the columns saved in the oneline data frame of the population:

In [ ]:
df_oneline = pd.read_hdf(local_file, key="oneline")

In [ ]:
df_oneline.columns

The oneline DataFrame stores a single summary row for each binary in the population exctracted by a POSYDON run. It retains the initial (`_i`) and final (`_f`) properties of the binary system and its two stellar components, together with quantities evaluated at key evolutionary stages (e.g., core helium depletion or core carbon depletion) that are required by certain supernova explodability prescriptions. In addition to the initial and final conditions, the `oneline` DataFrame also includes derived quantities such as `SN_type`.

For a detailed description of the oneline DataFrame and its contents, see the POSYDON documentation:
https://posydon.org/POSYDON/latest/tutorials-examples/population-synthesis/10_binaries_pop_syn.html#Population.oneline

**Important**: When creating your own population runs, ensure that the output columns required in the `.ini` file are properly specified. Many columns in both the `history` DataFrame and `oneline` DataFrame are not saved by default. For the `history` DataFrame, set the `only_select_columns` to include the desired attributes of the binary, star_1, and star_2 objects. For the oneline DataFrame, make sure the relevant scalar_names for these objects are included.


## 2.1 Distinguishing ECSNe from CCSNe

The column `SN_type`, stored under the `oneline` key, contains information about the mechanism of supernova event (ECSN, PISN, PPI, or iron core collapse) depending on the supernova prescription used.



**Note**: This is the mechanism of explosion, NOT the observational type

### Extracting the SN_type information stored in the `oneline` DataFrame

In [ ]:
df1_oneline = df_oneline["S1_SN_type"]
df1_oneline.rename('SN_type', inplace=True)

df2_oneline= df_oneline["S2_SN_type"]
df2_oneline.rename('SN_type', inplace=True)

df1_merged = df1.merge(
    df1_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)

df2_merged = df2.merge(
    df2_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)


df_synthetic_plus_oneline=pd.concat([df1_merged, df2_merged], axis=0)


We now print the outcome of the synthetic population, including the supernova (`SN_type`) information obtained from the `oneline` dataset.

In [ ]:
df_synthetic_plus_oneline

### Filter the transient population to show systems where star 1 or 2 produces an ECSN

In [ ]:
ecsn_pop = df_synthetic_plus_oneline[(df_synthetic_plus_oneline["SN_type"] == "ECSN")]
ecsn_pop.tail(5)

<div class="alert alert-success">

## Exercise: 
Calculate the fraction of neutron stars (NS) originating from ECSNe versus those from CCSNe, relative to the total NS population, and then visualize these fractions using a pie chart. (Replace the **??** values on the code below)
</div>

In [ ]:
# Count CCSNe
ALL_CCSN = (
    ( ?? ).sum() 
)

# Count ECSNe
ALL_ECSN = (
    ( ?? ).sum()
)

print("The number of all CCSNe is", ALL_CCSN)
print("The number of all ECSNe is", ALL_ECSN)

<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Solution </summary></b>
```python
ALL_ECSN = ((df_synthetic_plus_oneline["stellar_state_postCC"].isin(["NS"])) & (df_synthetic_plus_oneline["SN_type"] == "ECSN")).sum()
ALL_CCSN = ((df_synthetic_plus_oneline["stellar_state_postCC"].isin(["NS"])) & (df_synthetic_plus_oneline["SN_type"] == "CCSN")).sum()

print("The number of all CCSNe is", ALL_CCSN)
print("The number of all ECSNe is", ALL_ECSN)
```
    
</details>

After estimating the number of NS formed from CCSNe and ECSNe, visualize the results in a pie chart using the following code:

In [ ]:
labels = ['CCSN', 'ECSN']
sizes = [ALL_CCSN, ALL_ECSN]
colors = ['grey', 'green']



plt.figure(figsize=(4.5, 4.5))
wedges, texts, autotexts = plt.pie(
    sizes,
    labels=labels,
    colors=colors,
    autopct=lambda p: f"{p:.1f}%" if p > 0 else "",
    startangle=90,
    counterclock=False,
    wedgeprops={"edgecolor": "black", "linewidth": 0.5}
)


for autotext in autotexts:
    autotext.set_fontsize(14)
    autotext.set_color("black")

plt.axis('equal')  # Keep it circular
plt.title("Collapse mechanism", fontsize=8)
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise: 
Plot the initial mass distribution of single stars compared to the mass distribution of primary stars in binary systems that produce ECSNe, filtering the population as needed to retain the initial masses (fill in the **??**).
</div>

In [ ]:
## STEP 1: Extract the initial stellar masses and merge with oneline data ##

first_row = ?? # grab row corresponding to initial values
    
# extract the initial masses of the primary and single stars after masking    
df1['initial_mass'] = ??

#extract the initial masses of the secondary mass after masking
df2['initial_mass'] = ??


df1_merged = df1.merge(
    df1_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)

df2_merged = df2.merge(
    df2_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)

df_synthetic_plus_oneline=pd.concat([df1_merged, df2_merged], axis=0)

## STEP 2: Identify ECSNe from single stars and binary primaries ##
ECSN_from_primary = (

??    

)

ECSN_from_singles = (
    
??
    
)

# Extract the corresponding initial masses
masses_singles = ??
masses_primary = ??

## STEP 3: Plotting ##

import numpy as np

# Combine datasets
all_masses = ??

# Remove NaNs or infinities
all_masses = all_masses[np.isfinite(all_masses)]


bin_width = 0.5

if all_masses.min() == all_masses.max():
    bins = [all_masses.min() - bin_width / 2,
            all_masses.max() + bin_width / 2]
else:
    bins = np.arange(
        np.floor(all_masses.min()),
        np.ceil(all_masses.max()) + bin_width,
        bin_width,
    )

# Plot
plt.figure(figsize=(8,5))
plt.hist(masses_singles, bins=bins, alpha=0.6, label="Single Stars",
         color='skyblue', edgecolor='black', density=True)
plt.hist(masses_primary, bins=bins, alpha=0.6, label="Primary Stars",
         color='salmon', edgecolor='black', density=True)

plt.xlabel("$M_{1,initial}$ $(M_{\odot}$)")
plt.ylabel("Normalized Count")
plt.legend()
plt.show()

<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Hint 1  </summary></b>
```python
first_row= (df["step_names"] == "initial_cond") # grab row corresponding to initial values
    
# extract the initial masses of the primary and single stars after masking    
df1['initial_mass']=df['S1_mass'][first_row]

#extract the initial masses of the secondary mass after masking
df2['initial_mass']=df['S2_mass'][first_row]
```
    
</details>

<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Hint 2  </summary></b>
```python   
ECSN_from_primary = (
    (df_synthetic_plus_oneline["stellar_state_postCC"] == "NS")
    & (df_synthetic_plus_oneline["SN_type"] == "ECSN")
    & (df_synthetic_plus_oneline["binary_state_postCC"] != "initially_single_star")
    & (df_synthetic_plus_oneline["progenitor_star"] == 1)
)

ECSN_from_singles = (
    (df_synthetic_plus_oneline["stellar_state_postCC"] == "NS")
    & (df_synthetic_plus_oneline["SN_type"] == "ECSN")
    & (df_synthetic_plus_oneline["binary_state_postCC"] == "initially_single_star")
)

# Extract initial masses
masses_singles = df_synthetic_plus_oneline['initial_mass'][ECSN_from_singles]
masses_primary = df_synthetic_plus_oneline['initial_mass'][ECSN_from_primary]
```
    
</details>

<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Plotting  </summary></b>
```python
import numpy as np
# Combine datasets
all_masses = np.concatenate([masses_singles, masses_primary])

# Remove NaNs or infinities
all_masses = all_masses[np.isfinite(all_masses)]


bin_width = 0.5

if all_masses.min() == all_masses.max():
    bins = [all_masses.min() - bin_width / 2,
            all_masses.max() + bin_width / 2]
else:
    bins = np.arange(
        np.floor(all_masses.min()),
        np.ceil(all_masses.max()) + bin_width,
        bin_width,
    )

# Plot
plt.figure(figsize=(8,5))
plt.hist(masses_singles, bins=bins, alpha=0.6, label="Single Stars",
         color='skyblue', edgecolor='black', density=True)
plt.hist(masses_primary, bins=bins, alpha=0.6, label="Primary Stars",
         color='salmon', edgecolor='black', density=True)

plt.xlabel("$M_{1,initial}$ $(M_{\odot}$)")
plt.ylabel("Normalized Count")
plt.legend()
plt.show()
```
    
</details>

## 3. Observational SN types

### Classification of Observed SN Types

We separate **H-poor** (Ib, Ic, IIb) from **H-rich** (II) supernovae using hydrogen ejecta mass and surface nitrogen and helium abundances. 
To classify pre-CC models as progenitors of SNe II, IIb, Ib, or Ic, we use the total H mass in the ejecta $(M_{H,ej})$ as a primary indicator, along with the pre-SN surface He4 and N14 abundances evaluated at the time of core carbon depletion. However, the relationship between the structure of the pre-SN progenitor and the spectroscopic characteristics of the explosion remains an active area of research. Here we will just use some criteria that are suggested in the literature:


#### Assumed Criteria based on literature 
(Gilkis et al. 2019, Yoon et al. 2017; Sravan et al. 2018, Aguilera-Dena et al. 2023, Dessart et al. 2020)
- **Ic:** $~M_{H,ej}$ < 0.033 $M_{\odot}$ $~\&~$  $X_{N,surf}$ < 1e-4 & $~\&~$  $X_{He4, surf}$ < 0.5
- **Ib:** $~M_{H,ej}$ < 0.033 $M_{\odot}$ $~\&~$  $X_{N,surf}$ >= 1e-4 & $~OR~$  $X_{He4, surf}$ >= 0.5
- **IIb:**  0.033 $M_{\odot}$ <= $M_{H,ej}$ <= 0.5 $M_{\odot}$
- **II:** $~M_{H,ej}$ > 0.5 $M_{\odot}$

#### Color Convention
- <span style="color:cyan; font-weight:bold">Type Ic</span> — cyan 
- <span style="color:blue; font-weight:bold">Type Ib</span> — blue  
- <span style="color:gold; font-weight:bold">Type IIb</span> — yellow  
- <span style="color:red; font-weight:bold">Type II</span> — red  


Below is a function we created to classify each SN event as Ic, Ib, IIb or II. Please review it carefully to understand the parameters and the columns used. This time, we also need to retain the surface nitrogen and helium abundances at the time of core carbon depletion as well as the hydrogen mass ejecta during SNe,  and we have to modify the population dataframe accordingly.

In [ ]:
def classify_observed_SN(
    df,
    M_H_Ib=0.033,      # [Msun] Hydrogen ejecta threshold between Ib/Ic and IIb (Gilkis+2019)
    M_H_II=0.5,        # [Msun] Hydrogen ejecta threshold between IIb and II (Yoon+2017; Sravan+2018)
    N_surf_Ic=1e-4,    # [-] Surface nitrogen threshold to distinguish Ic from Ib (Aguilera-Dena+2023)
    he4_surf_Ic= 0.5,   # [-] Surface nitrogen threshold to distinguish Ic from Ib (Aguilera-Dena+2023)
    m_col="h1_mass_ej",  # Column containing hydrogen ejecta mass
    n_col="surface_n14",  # Column containing surface nitrogen abundance
    he4_col="surface_he4",  # Column containing surface helium abundance
    state="stellar_state_postCC"  # Column containing the state of the star after CC1 or CC2

):
    """
    

    The classification is based on the hydrogen ejecta mass (M) and surface 
    nitrogen abundance (N), surface Helium abundance (He4) using thresholds from the literature.

    Rules:
      - Type Ic :  M < M_H_Ib  and  N <  N_surf_Ic and h < he4_surf_Ic
      - Type Ib :  M < M_H_Ib  and  (N >= N_surf_Ic OR h >= he4_surf_Ic)
      - Type IIb:  M_H_Ib < M < M_H_II
      - Type II :  M >= M_H_II
      - Otherwise: "Unknown"

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe, one row per progenitor.
    M_H_Ib : float
        Threshold hydrogen ejecta mass (Msun) separating Ib/Ic from IIb.
    M_H_II : float
        Threshold hydrogen ejecta mass (Msun) separating IIb from II.
    N_surf_Ic : float
        Threshold surface nitrogen abundance distinguishing Ic from Ib.
    m_col : str
        Name of the column in df containing hydrogen ejecta mass.
    n_col : str
        Name of the column in df containing surface nitrogen abundance.

    Returns
    -------
    df : pandas.DataFrame
        Dataframe with one new column:
          - "SN_observed": the SN subtype (II, IIb, Ib, Ic, or Unknown).
    """

    # --- Check that required columns exist ---
    if m_col not in df.columns or n_col not in df.columns:
        raise KeyError(f"Expected columns '{m_col}' and '{n_col}' in df.")

    # Extract relevant quantities
    M = df[m_col]   # hydrogen ejecta mass
    N = df[n_col]   # surface nitrogen abundance
    h = df[he4_col] # surface helium abundance
    s= df[state] # post-CC state

    # Start with everything labeled as Unknown
    observed_type = df.index.to_series().map(lambda _: "Unknown")

    # Apply classification rules
    is_Ic  = (M < M_H_Ib) & (N <  N_surf_Ic) & (h <  he4_surf_Ic) & (s=="NS")
    is_Ib  = (M < M_H_Ib) & ((N >= N_surf_Ic) | (h >=  he4_surf_Ic)) & (s=="NS")
    is_IIb = (M >= M_H_Ib) & (M <  M_H_II) & (s=="NS")
    is_II  = (M >= M_H_II) & (s=="NS")

    observed_type = observed_type.mask(is_Ic,  "Ic")
    observed_type = observed_type.mask(is_Ib,  "Ib")
    observed_type = observed_type.mask(is_IIb, "IIb")
    observed_type = observed_type.mask(is_II,  "II")

    # Save results back into the dataframe
    df["SN_observed"] = observed_type

    return df


<div class="alert alert-success">

## Exercise: Find the relative rates of all SN Types in the population

1. Based on the assumed classification criteria above and the inputs used in the `classify_observed_SN`, determine which properties and columns need to be retained.

2. Replace the **??** values in the code below and create the new population keeping the information that should be used in the `classify_observed_SN` function.
   
</div>

<div class="alert alert-warning" style="margin-top: 20px">

<details>
<summary><b>Hint 1</b></summary>
Some properties/columns describe the preCC (progenitor) state, others pertain to the compact object, and a few are listed directly in the outline since they were pre-calculated from the profile.
</details>

<div class="alert alert-warning" style="margin-top: 20px">

<details>
<summary><b>Hint2</b></summary>
We should modify the script below to include information of the surface abundances of the stars 
before SNe (at core carbon depletion) adding the appropriate columns.

- **preSN** line → progenitor properties (e.g., surface abundances, core mass etc). In our case we need `surface_he4`,`surface_n14`, preSN mass of the progenitor
- **postSN** line → compact object mass and state, orbit re-adjustment after instataneous mass loss and natal kick, etc. In our case we need compact object mass and state
- **oneline** dataframe → keeping some usually pre_computed info about the event. In our case we need `h1_mass_ej`

- </details>

In [ ]:
# -----------------------------------------------------------------------------
# Identify the rows corresponding to the core-collapse events of star 1 and star 2.
# These rows contain the stellar properties immediately before the supernova.

pre_CC1 = (df['event'] == ?? )
pre_CC2 = (df['event'] == "CC2")


# -----------------------------------------------------------------------------
# Extract the surface composition of the progenitor star before collapse.
# For star 1 we keep the surface abundances of 14N and 4He.

df1_pre = df[[??, ??]][pre_CC1].copy()

# Rename columns to remove the star identifier since this dataframe only
# contains the progenitor information for the collapsing star.

df1_pre = df1_pre.rename(columns={
    ??: 'surface_n14',
    ??: 'surface_he4'
})


# Add the pre-supernova surface properties to the post-core-collapse dataframe.
# The index is used to match each supernova event with its progenitor properties.

df1_n = pd.merge(
    df1,
    df1_pre,
    left_index=True,
    right_index=True,
    how='inner'
)


# -----------------------------------------------------------------------------
# Repeat the same procedure for star 2.

df2_pre = df[[??, ??]][??].copy()

df2_pre = df2_pre.rename(columns={
    ??: 'surface_n14',
    ??: 'surface_he4'
})

df2_n = pd.merge(
    df2,
    df2_pre,
    left_index=True,
    right_index=True,
    how='inner'
)


# -----------------------------------------------------------------------------
# Extract additional supernova information from the oneline dataframe.
# These quantities provide information about the SN explosion properties,
# such as the SN type and the amount of hydrogen mass ejected.

# Information for the primary star (star 1)
df1_oneline = df_oneline[['S1_SN_type', ??]].copy()

df1_oneline = df1_oneline.rename(columns={
    'S1_SN_type': 'SN_type',
    ??: 'h1_mass_ej'
})


# Information for the secondary star (star 2)
df2_oneline = df_oneline[['S2_SN_type', ??]].copy()

df2_oneline = df2_oneline.rename(columns={
    'S2_SN_type': 'SN_type',
    ??: 'h1_mass_ej'
})


# -----------------------------------------------------------------------------
# Combine the core-collapse dataframe with the oneline SN properties.
# The index ensures that the correct binary system is matched.

df1_merged = df1_n.merge(
    df1_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)

df2_merged = df2_n.merge(
    df2_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)


# -----------------------------------------------------------------------------
# Combine the explosions from the primary and secondary stars into one dataframe.
# Each row represents a supernova event and contains:
#   - compact remnant properties
#   - binary state after collapse
#   - progenitor surface composition
#   - SN explosion properties

df_synthetic_plus_oneline_plus_pre_CC = pd.concat(
    [df1_merged, df2_merged],
    axis=0
)

<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Solution  </summary></b>
```python

# -----------------------------------------------------------------------------
# Identify the rows corresponding to the core-collapse events of star 1 and star 2.
# These rows contain the stellar properties immediately before the supernova.

pre_CC1 = (df['event'] == "CC1")
pre_CC2 = (df['event'] == "CC2")


# -----------------------------------------------------------------------------
# Extract the surface composition of the progenitor star before collapse.
# For star 1 we keep the surface abundances of 14N and 4He.

df1_pre = df[['S1_surface_n14', 'S1_surface_he4']][pre_CC1].copy()

# Rename columns to remove the star identifier since this dataframe only
# contains the progenitor information for the collapsing star.
    
df1_pre = df1_pre.rename(columns={
    'S1_surface_n14': 'surface_n14',
    'S1_surface_he4': 'surface_he4'
})


# Add the pre-supernova surface properties to the post-core-collapse dataframe.
# The index is used to match each supernova event with its progenitor properties.
    
df1_n = pd.merge(
    df1,
    df1_pre,
    left_index=True,
    right_index=True,
    how='inner'
)


# -----------------------------------------------------------------------------
# Repeat the same procedure for star 2.

    
df2_pre = df[['S2_surface_he4', 'S2_surface_n14']][pre_CC2].copy()

df2_pre = df2_pre.rename(columns={
    'S2_surface_n14': 'surface_n14',
    'S2_surface_he4': 'surface_he4'
})

df2_n = pd.merge(
    df2,
    df2_pre,
    left_index=True,
    right_index=True,
    how='inner'
)


# -----------------------------------------------------------------------------
# Extract additional supernova information from the oneline dataframe.
# These quantities provide information about the SN explosion properties,
# such as the SN type and the amount of hydrogen mass ejected.

# Information for the primary star (star 1)
df1_oneline = df_oneline[['S1_SN_type', 'S1_h1_mass_ej']].copy()

df1_oneline = df1_oneline.rename(columns={
    'S1_SN_type': 'SN_type',
    'S1_h1_mass_ej': 'h1_mass_ej'
})


# Information for the secondary star (star 2)
df2_oneline = df_oneline[['S2_SN_type', 'S2_h1_mass_ej']].copy()

df2_oneline = df2_oneline.rename(columns={
    'S2_SN_type': 'SN_type',
    'S2_h1_mass_ej': 'h1_mass_ej'
})


# -----------------------------------------------------------------------------
# Combine the core-collapse dataframe with the oneline SN properties.
# The index ensures that the correct binary system is matched.
    
df1_merged = df1_n.merge(
    df1_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)

df2_merged = df2_n.merge(
    df2_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)


# -----------------------------------------------------------------------------
# Combine the explosions from the primary and secondary stars into one dataframe.
# Each row represents a supernova event and contains:
#   - compact remnant properties
#   - binary state after collapse
#   - progenitor surface composition
#   - SN explosion properties
    
df_synthetic_plus_oneline_plus_pre_CC = pd.concat(
    [df1_merged, df2_merged],
    axis=0
)

```
    
</details>

In [ ]:
df_synthetic_plus_oneline_plus_pre_CC 

Estimate the relative rates of different observational supernova (SN) types. Absolute rates would require normalization by the mass formed in the stellar populations (see this afternoon’s lecture for details).

In [ ]:
import matplotlib.pyplot as plt

# Classify observed SN types with your chosen thresholds
df_classified = classify_observed_SN(df_synthetic_plus_oneline_plus_pre_CC, M_H_Ib=0.033, M_H_II=0.5, N_surf_Ic=1e-4)

# Collect observed SN types for both stars
obs = df_classified["SN_observed"]
obs = obs[obs.isin(["Ic", "Ib", "IIb", "II"])]  # drop Unknown if present

# Count and normalize
counts = obs.value_counts().reindex(["Ic", "Ib", "IIb", "II"], fill_value=0)
fractions = counts.values.astype(float)
fractions = fractions / fractions.sum()

# Labels and colors
labels = ["Type Ic", "Type Ib", "Type IIb", "Type II"]
colors = ["cyan", "blue", "yellow", "red"]
explode = [0.05] * 4

# Plot pie chart
plt.figure(figsize=(4.0, 4.0))
wedges, texts, autotexts = plt.pie(
    fractions,
    labels=labels,
    colors=colors,
    explode=explode,
    autopct=lambda p: f"{p:.1f}%" if p > 0 else "",
    startangle=90,
    counterclock=False,
    wedgeprops={"edgecolor": "black", "linewidth": 0.5}
)

for autotext in autotexts:
    autotext.set_fontsize(7)
    autotext.set_color("black")

plt.title("Fraction of Observed Supernova Types")
plt.show()


## 4. Total ejecta masses

<div class="alert alert-success">


## Exercise: Calculate the distribution of ejecta masses of SNe. 

Let's stick to **core-collapse SNe**  and to events that form **neutron stars** only, to not complicate it with potential fallback from black holes and for Observed **Type Ib** SNe only. Fill in the **??** placeholders in the code below.

   
</div>

<div class="alert alert-warning" style="margin-top: 20px">

<details>
<summary><b>Hint</b></summary>

The ejecta mass is given by  

$
M_{\mathrm{ej}} = M_{\mathrm{prog}} - M_{\mathrm{rem}}
\$

where $(M_{\mathrm{prog}})$ is the progenitor mass at core carbon depletion, and $(M_{\mathrm{rem}})$ is the mass of the neutron star.  

</details>


First we will keep the information of preSN mass of the stellar progenitor

In [ ]:

# -----------------------------------------------------------------------------
# Identify the rows corresponding to the core-collapse events of star 1 and star 2.
# These rows contain the stellar properties immediately before the supernova.

pre_CC1 = ??
pre_CC2 = ??


# -----------------------------------------------------------------------------
# Extract the surface composition and the mass of the progenitor star before collapse.
# For star 1 we keep the surface abundances of 14N and 4He.

df1_pre = df[['S1_surface_n14', 'S1_surface_he4', ??]][pre_CC1].copy()

# Rename columns to remove the star identifier since this dataframe only
# contains the progenitor information for the collapsing star.
df1_pre = df1_pre.rename(columns={
    'S1_surface_n14': 'surface_n14',
    'S1_surface_he4': 'surface_he4',
    ?? : 'preCC_mass'
})


# Add the pre-supernova surface properties to the post-core-collapse dataframe.
# The index is used to match each supernova event with its progenitor properties.
df1_n = pd.merge(
    df1,
    df1_pre,
    left_index=True,
    right_index=True,
    how='inner'
)


# -----------------------------------------------------------------------------
# Repeat the same procedure for star 2.

df2_pre = df[['S2_surface_he4', 'S2_surface_n14', ??]][pre_CC2].copy()

df2_pre = df2_pre.rename(columns={
    'S2_surface_n14': 'surface_n14',
    'S2_surface_he4': 'surface_he4',
    ?? : 'preCC_mass'

    
})

df2_n = pd.merge(
    df2,
    df2_pre,
    left_index=True,
    right_index=True,
    how='inner'
)


# -----------------------------------------------------------------------------
# Extract additional supernova information from the oneline dataframe.
# These quantities provide information about the SN explosion properties,
# such as the SN type and the amount of hydrogen mass ejected.


# Information for the primary star (star 1)
df1_oneline = df_oneline[['S1_SN_type', 'S1_h1_mass_ej']].copy()

df1_oneline = df1_oneline.rename(columns={
    'S1_SN_type': 'SN_type',
    'S1_h1_mass_ej': 'h1_mass_ej'
})


# Information for the secondary star (star 2)
df2_oneline = df_oneline[['S2_SN_type', 'S2_h1_mass_ej']].copy()

df2_oneline = df2_oneline.rename(columns={
    'S2_SN_type': 'SN_type',
    'S2_h1_mass_ej': 'h1_mass_ej'
})


# -----------------------------------------------------------------------------
# Combine the core-collapse dataframe with the oneline SN properties.
# The index ensures that the correct binary system is matched.

df1_merged = df1_n.merge(
    df1_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)

df2_merged = df2_n.merge(
    df2_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)


# -----------------------------------------------------------------------------
# Combine the explosions from the primary and secondary stars into one dataframe.
# Each row represents a supernova event and contains:
#   - compact remnant properties
#   - binary state after collapse
#   - progenitor surface composition
#   - SN explosion properties

df_synthetic_plus_oneline_plus_pre_CC_ejecta = pd.concat(
    [df1_merged, df2_merged],
    axis=0
)

<div class="alert alert-warning" style="margin-top: 20px">
<details>

<b><summary>Solution  </summary></b>
```

# -----------------------------------------------------------------------------
# Identify the rows corresponding to the core-collapse events of star 1 and star 2.
# These rows contain the stellar properties immediately before the supernova.
# -----------------------------------------------------------------------------
pre_CC1 = (df['event'] == "CC1") 
pre_CC2 = (df['event'] == "CC2") 


# -----------------------------------------------------------------------------
# Extract the surface composition and the mass of the progenitor star before collapse.
# For star 1 we keep the surface abundances of 14N and 4He.
# -----------------------------------------------------------------------------
df1_pre = df[['S1_surface_n14', 'S1_surface_he4', 'S1_mass']][pre_CC1].copy()

# Rename columns to remove the star identifier since this dataframe only
# contains the progenitor information for the collapsing star.
df1_pre = df1_pre.rename(columns={
    'S1_surface_n14': 'surface_n14',
    'S1_surface_he4': 'surface_he4',
    'S1_mass': 'preCC_mass'
})


# Add the pre-supernova surface properties to the post-core-collapse dataframe.
# The index is used to match each supernova event with its progenitor properties.
df1_n = pd.merge(
    df1,
    df1_pre,
    left_index=True,
    right_index=True,
    how='inner'
)


# -----------------------------------------------------------------------------
# Repeat the same procedure for star 2.
# -----------------------------------------------------------------------------
df2_pre = df[['S2_surface_he4', 'S2_surface_n14', 'S2_mass']][pre_CC2].copy()

df2_pre = df2_pre.rename(columns={
    'S2_surface_n14': 'surface_n14',
    'S2_surface_he4': 'surface_he4',
    'S2_mass': 'preCC_mass'

    
})

df2_n = pd.merge(
    df2,
    df2_pre,
    left_index=True,
    right_index=True,
    how='inner'
)


# -----------------------------------------------------------------------------
# Extract additional supernova information from the oneline dataframe.
# These quantities provide information about the SN explosion properties,
# such as the SN type and the amount of hydrogen mass ejected.
# -----------------------------------------------------------------------------

# Information for the primary star (star 1)
df1_oneline = df_oneline[['S1_SN_type', 'S1_h1_mass_ej']].copy()

df1_oneline = df1_oneline.rename(columns={
    'S1_SN_type': 'SN_type',
    'S1_h1_mass_ej': 'h1_mass_ej'
})


# Information for the secondary star (star 2)
df2_oneline = df_oneline[['S2_SN_type', 'S2_h1_mass_ej']].copy()

df2_oneline = df2_oneline.rename(columns={
    'S2_SN_type': 'SN_type',
    'S2_h1_mass_ej': 'h1_mass_ej'
})


# -----------------------------------------------------------------------------
# Combine the core-collapse dataframe with the oneline SN properties.
# The index ensures that the correct binary system is matched.
# -----------------------------------------------------------------------------
df1_merged = df1_n.merge(
    df1_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)

df2_merged = df2_n.merge(
    df2_oneline,
    left_index=True,
    right_index=True,
    how="inner"
)


# -----------------------------------------------------------------------------
# Combine the explosions from the primary and secondary stars into one dataframe.
# Each row represents a supernova event and contains:
#   - compact remnant properties
#   - binary state after collapse
#   - progenitor surface composition
#   - SN explosion properties
# -----------------------------------------------------------------------------
df_synthetic_plus_oneline_plus_pre_CC_ejecta = pd.concat(
    [df1_merged, df2_merged],
    axis=0
)

    

```
</details>

### Estimating the ejecta masses of CC1 and CC2 for type Ib SNe


In [ ]:
df_classified 

In [ ]:
# Run classification on the underlying DataFrame
df_classified = classify_observed_SN(
    df_synthetic_plus_oneline_plus_pre_CC_ejecta,
    M_H_Ib=0.033, M_H_II=0.5, N_surf_Ic=1e-4
)

# Compute ejecta mass = preSN mass - remnant mass  # CHANGED: explicitly add ejecta columns
df_classified["M_ejecta"] = df_classified["preCC_mass"] - df_classified["compact_object_mass"]

is_NS = df_classified['stellar_state_postCC'].eq('NS')

# Observed Type Ib masks
is_Ib = df_classified['SN_observed'].eq('Ib')

# Ejecta masses for Ib + NS only, positive values
M_ej_Ib_NS = pd.concat([
    df_classified.loc[is_NS & is_Ib, 'M_ejecta'],
]).dropna()
M_ej_Ib_NS = M_ej_Ib_NS[M_ej_Ib_NS > 0]

# Plot
plt.figure(figsize=(6,6))
plt.hist(M_ej_Ib_NS, bins=15, alpha=0.85)
plt.xlabel(r"$M_{\mathrm{ej}}$ [M$_\odot$]")
plt.ylabel("Count")
plt.title("Ejecta masses: Type Ib CCSNe with NS remnants")
plt.tight_layout()
plt.show()


### Finding which channels are the most dominant ones contributing to Type Ib

This section covers how to obtain information about the evolutionary history of binary systems prior to their collapse and display the Van den Heuvel diagram

In [ ]:
df_channels = pd.read_hdf(local_file, key="formation_channels")

In [ ]:
df_synthetic_with_channels = df_synthetic_plus_oneline_plus_pre_CC_ejecta.merge(
    df_channels,
    left_index=True,
    right_index=True,
    how="inner"
)

<div class="alert alert-success">

## Exercise:
Identify the evolutionary channels of star 1 and star 2 that can lead to the production of Type Ib supernovae. Then, determine from the most dominant evolutionary path one binary parameter index and use it to plot the Van den Heuvel diagrams for this system.

   
</div>



In [ ]:
df_classified = classify_observed_SN(
    df_synthetic_with_channels,
    M_H_Ib=0.033, M_H_II=0.5, N_surf_Ic=1e-4
)

In [ ]:
df_classified

In [ ]:
typeIb_SNe_from_star1 = ((df_classified['stellar_state_postCC']== 'NS') & 
        (df_classified['SN_observed']== 'Ib') &
        (df_classified['progenitor_star']==1) &
        (df_classified['binary_state_postCC']!='initially_single_star'))

df_classified['channel_debug'][typeIb_SNe_from_star1].value_counts()


In [ ]:
typeIb_SNe_from_star1 = ((df_classified['stellar_state_postCC']== 'NS') & 
        (df_classified['SN_observed']== 'Ib') &
        (df_classified['progenitor_star']==2) &
        (df_classified['binary_state_postCC']!='initially_single_star'))

df_classified['channel_debug'][typeIb_SNe_from_star1].value_counts()


If you install POSYDON on your computer following the instructions provided in the linked guide, you’ll have the option to enable experimental visualization libraries. While these libraries offer advanced features, please note that they might still be in development and could be subject to changes.
Instructions for visualizations: https://posydon.org/POSYDON/latest/getting-started/installation-guide.html#id11

To install these experimental visualization libraries
Navigate to your POSYDON directory (where the `setup.py` is located) and run:
`pip install ".[vis]"`

Unfortunataly, this will not work in the environemnt for the School. 
See the example commands below. TAs will show an example diagram.

```python
from posydon.visualization.VHdiagram import VHdiagram

VHdiagram('1e+00_Zsun_population.h5', path='.', index=4)
```

```python
from posydon.visualization.VHdiagram import DisplayMode
from posydon.visualization.VH_diagram.Presenter import PresenterMode

VHdiagram(
    '1e+00_Zsun_population.h5', 
    path='.',
    index=21,
    presentMode=PresenterMode.DIAGRAM,
    displayMode=DisplayMode.INLINE_B,
)
```

#### If you’re efficient and have time left, repeat the process for secondaries to identify the most dominant channel producing type Ic SNe.